# Batch / Tiled Inference on Large GeoTIFFs

SAM processes images at a fixed resolution (1024×1024 px internally). For large orthophotos or satellite scenes, samgeo provides `tiff_to_tiff()` which tiles the input, runs SAM on each tile, and reassembles the results into a single output GeoTIFF.

This notebook covers:
1. Running tiled inference with `SamGeo.tiff_to_tiff()`
2. Controlling tile size and overlap
3. Exporting the merged output as a Cloud-Optimized GeoTIFF (COG)

## References
- samgeo tiff_to_tiff: https://samgeo.gishub.org/samgeo/#samgeo.samgeo.SamGeo.tiff_to_tiff

## 1. Setup

In [ ]:
import os
import urllib.request
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show
from samgeo import SamGeo

url = "https://github.com/opengeos/samgeo/releases/download/v0.2.0/aerial_image.tif"
image_path = "aerial_image.tif"
if not os.path.exists(image_path):
    urllib.request.urlretrieve(url, image_path)

with rasterio.open(image_path) as src:
    print(f"Image size: {src.width} x {src.height} px")
    print(f"CRS: {src.crs}")

## 2. Tiled Automatic Segmentation

`tiff_to_tiff()` handles tiling internally. The `batch_size` parameter controls how many tiles are processed at once.

In [ ]:
sam = SamGeo(
    model_type="vit_b",
    automatic=True,
    points_per_side=16,
    pred_iou_thresh=0.86,
    stability_score_thresh=0.92,
    min_mask_region_area=50,
)

sam.generate(
    source=image_path,
    output="tiled_masks.tif",
    foreground=True,
    unique=True,
)
print("Tiled segmentation complete.")

## 3. Inspect Output

In [ ]:
import numpy as np

with rasterio.open("tiled_masks.tif") as src:
    arr = src.read(1)
    print(f"Output shape: {arr.shape}")
    print(f"Output CRS: {src.crs}")
    n_objects = len(np.unique(arr)) - 1  # subtract background
    print(f"Objects detected: {n_objects}")

## 4. Visualize Merged Output

In [ ]:
sam.show_anns(
    cmap="tab20",
    alpha=0.45,
    title="Tiled SAM segmentation",
    blend=True,
    output="tiled_viz.png",
)

img = plt.imread("tiled_viz.png")
plt.figure(figsize=(10, 10))
plt.imshow(img)
plt.axis("off")
plt.tight_layout()
plt.show()

## 5. Export as Cloud-Optimized GeoTIFF (COG)

COG format allows efficient streaming access and is compatible with GIS tools like QGIS and ArcGIS.

In [ ]:
import subprocess

# Requires gdal_translate (available via system GDAL or conda-forge)
result = subprocess.run(
    [
        "gdal_translate",
        "-of", "COG",
        "-co", "COMPRESS=LZW",
        "tiled_masks.tif",
        "tiled_masks_cog.tif",
    ],
    capture_output=True,
    text=True,
)
if result.returncode == 0:
    print("COG export successful: tiled_masks_cog.tif")
else:
    print("gdal_translate not available. Install GDAL or use rasterio to convert:")
    print("  pip install gdal")
    print(result.stderr)

## 6. Vector Export (GeoJSON / Shapefile)

In [ ]:
sam.tiff_to_vector("tiled_masks.tif", "tiled_masks.geojson")
print("Vector polygons exported to tiled_masks.geojson")

## Memory and Performance Notes

- `vit_b` uses ~2 GB VRAM / RAM; `vit_h` uses ~8 GB.
- For very large scenes (>10k × 10k px), process in tiles using a sliding window and merge results afterward.
- CPU-only inference is supported but slow (~30–120 s per tile depending on hardware and model size).